# IndicConformer (Punjabi) test on Google Colab

**Why this notebook exists:** we tried loading AI4Bharat's `indicconformer_stt_pa_hybrid_ctc_rnnt_large` model
locally on Windows using the public `pip install nemo_toolkit[asr]` package, and hit a hard blocker: this
checkpoint's tokenizer config uses a `multilingual` type that the public NeMo package does not implement -
only AI4Bharat's own fork of NeMo supports it. This notebook installs that fork instead (Colab's Linux+GPU
environment is what AI4Bharat actually develops/tests it against), loads the model, transcribes a test
Punjabi audio clip, and compares the output against our current production fine-tuned Whisper model.

**Before running:** go to `Runtime > Change runtime type` and select a **GPU** (T4 is fine) - installing/running
NeMo on CPU-only is much slower and the `reinstall.sh` step can take a long time either way.

**You will also need:** a Hugging Face account that has clicked "Agree and access repository" on
https://huggingface.co/ai4bharat/indicconformer_stt_pa_hybrid_ctc_rnnt_large (gated repo), and a HF access
token (from https://huggingface.co/settings/tokens) to paste into the login cell below.

In [ ]:
!nvidia-smi

## 1. Install AI4Bharat's NeMo fork (NOT the public `nemo_toolkit` package)

This is the step that's impossible to skip - the public package genuinely lacks the multilingual tokenizer
support this model needs. This can take several minutes.

In [ ]:
!python --version
!git clone https://github.com/AI4Bharat/NeMo.git
%cd NeMo
!git checkout nemo-v2
# Install only the ASR extra instead of running reinstall.sh (which installs ".[all]" -
# NLP/TTS/multimodal extras too). Those pull in tensorstore/typed-ast, which don't have
# prebuilt wheels for Colab's current Python and fail to build from source. We only
# need ASR here, so scoping the install avoids that dependency entirely.
!pip install --editable ".[asr]"
%cd ..

In [ ]:
!pip install --quiet gTTS huggingface_hub faster-whisper
!apt-get -y -qq install ffmpeg

## 2. Log in to Hugging Face (required for the gated IndicConformer repo)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Download the IndicConformer checkpoint

In [ ]:
from huggingface_hub import hf_hub_download

indicconformer_path = hf_hub_download(
    repo_id="ai4bharat/indicconformer_stt_pa_hybrid_ctc_rnnt_large",
    filename="indicconformer_stt_pa_hybrid_rnnt_large.nemo",
)
print(indicconformer_path)

## 4. Generate a real Punjabi test clip (same phrase used in the local test)

In [ ]:
from gtts import gTTS

text = "\u0a38\u0a24 \u0a38\u0a4d\u0a30\u0a40 \u0a05\u0a15\u0a3e\u0a32, \u0a24\u0a41\u0a38\u0a40\u0a02 \u0a15\u0a3f\u0a35\u0a47\u0a02 \u0a39\u0a4b?"
gTTS(text=text, lang="pa").save("punjabi_test.mp3")
!ffmpeg -y -loglevel error -i punjabi_test.mp3 -ar 16000 -ac 1 punjabi_test_16k.wav
print("Test phrase:", text)

## 5. Load IndicConformer and transcribe

In [ ]:
import nemo.collections.asr as nemo_asr

model = nemo_asr.models.ASRModel.restore_from(restore_path=indicconformer_path)
model.eval()
model.cur_decoder = "rnnt"  # switch to "ctc" to compare the CTC head instead

# IndicConformer is multilingual with per-language decoder heads keyed by language code,
# so language_id must be passed explicitly or the internal lookup receives None.
indicconformer_result = model.transcribe(["punjabi_test_16k.wav"], batch_size=1, language_id="pa")
print("IndicConformer transcript:", indicconformer_result)

## 6. For comparison: transcribe the same clip with our current production model

This pulls the exact fine-tuned Whisper CT2 backup from `AI-pro11/whisper-punjabi-ct2` (one of our
confirmed backup locations) so we can compare outputs side by side on identical audio.

In [ ]:
from huggingface_hub import snapshot_download
from faster_whisper import WhisperModel

current_model_dir = snapshot_download(repo_id="AI-pro11/whisper-punjabi-ct2")
current_model = WhisperModel(current_model_dir, device="cpu", compute_type="int8")

segments, info = current_model.transcribe("punjabi_test_16k.wav", beam_size=1, vad_filter=True)
current_model_text = "".join(seg.text for seg in segments)
print("Current production model transcript:", current_model_text)

## 7. Side-by-side summary

In [ ]:
print("Reference text:               ", text)
print("IndicConformer transcript:    ", indicconformer_result)
print("Current production transcript:", current_model_text)

**Next steps once this runs successfully:** try additional, more varied Punjabi audio (ideally real recorded
speech, not just gTTS) before deciding whether to actually replace the production model. Nothing in
production (`ai_backend_server.py`, `Dockerfile`, `fly.toml`) should change until this comparison looks
convincingly better across multiple samples.